# 🐍 Python para Ingeniería de Datos — BSG Institute
## Sesión 11/12 · Bloques 4.8.x
### Docker + Proyecto Final

---

**Lo que veremos hoy:**
1. ¿Qué problema resuelve Docker?
2. Conceptos clave: imagen, contenedor, capa, puerto
3. Anatomía del `Dockerfile` de nuestra API
4. Construir y correr el contenedor paso a paso
5. Proyecto final: dockerizar la API + personalizar el dashboard

---

> ⚠️ **Prerequisito:** Docker Desktop instalado y corriendo.
> Verifica en la terminal: `docker --version`
> Si no lo tienes, avisa al instructor ahora.

---
## 🧠 CONCEPTO: ¿Qué problema resuelve Docker?

Construiste un pipeline que funciona perfecto en tu máquina:
- Python 3.12 instalado
- Las librerías exactas en el venv
- El archivo `.env` con las credenciales
- La carpeta en la ruta correcta

Ahora imagina que lo tienes que entregar a:
- Un colega que tiene Python 3.9
- Un servidor en la nube con Ubuntu sin nada instalado
- El cliente que tiene Windows y nunca ha instalado pip

**El problema:** *"En mi máquina funciona"* — la frase más temida en ingeniería de software.

> 💡 **Analogía:** Docker es como empacar toda tu cocina — las ollas, los ingredientes, la receta y hasta la estufa — en una caja estandarizada. Donde abras esa caja, la comida sale igual. No importa si estás en México, Alemania o un servidor en la nube.

**Los tres conceptos clave:**

| Concepto | Qué es | Analogía |
|---|---|---|
| **Imagen** | Plantilla con todo lo necesario | La receta + ingredientes sellados |
| **Contenedor** | Una imagen corriendo | El platillo ya servido |
| **Dockerfile** | Instrucciones para construir la imagen | La receta paso a paso |

---
## ✅ CELDA 1 — Verificar Docker

In [ ]:
import subprocess

def run(cmd):
    """Ejecuta un comando de terminal y muestra el resultado."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    output = result.stdout.strip() or result.stderr.strip()
    print(output)
    return result.returncode

print('=== Versión de Docker ===')
run('docker --version')

print('\n=== Docker está corriendo? ===')
code = run('docker info --format "Docker Engine: {{.ServerVersion}}"')
if code == 0:
    print('✅ Docker Desktop está activo')
else:
    print('❌ Docker no está corriendo — abre Docker Desktop e intenta de nuevo')

---
## 📄 CELDA 2 — Anatomía del Dockerfile

Antes de construir, leemos y entendemos cada instrucción del `Dockerfile`.

In [ ]:
with open('Dockerfile', encoding='utf-8') as f:
    contenido = f.read()

print(contenido)

---
### Las capas del Dockerfile

Cada instrucción (`FROM`, `WORKDIR`, `COPY`, `RUN`, `CMD`) crea una **capa**.
Docker guarda cada capa en cache — si una capa no cambió, no la reconstruye.

```
FROM python:3.12-slim     ← Capa 1: imagen base (se descarga una vez)
WORKDIR /app              ← Capa 2: directorio (instantáneo)
COPY requirements_api.txt ← Capa 3: el archivo de dependencias
RUN pip install ...       ← Capa 4: instala librerías (tarda ~1 min la 1a vez)
COPY api.py               ← Capa 5: tu código
CMD ["uvicorn", ...]      ← No es capa — es el comando de inicio
```

> 💡 **Por qué copiamos `requirements_api.txt` ANTES que `api.py`:**
> Si cambias el código de `api.py`, Docker solo reconstruye la Capa 5.
> Las dependencias (Capa 4, la más lenta) quedan en cache.
> Si los pusieras al revés, reinstalaría todo cada vez que cambias una línea de código.

---
## 🔧 CELDA 3 — Verificar archivos necesarios antes de construir

In [ ]:
import os

archivos_necesarios = [
    ('Dockerfile',            'Instrucciones de la imagen'),
    ('.dockerignore',         'Archivos excluidos de la imagen'),
    ('api.py',                'La API a dockerizar'),
    ('requirements_api.txt',  'Dependencias de la imagen'),
    ('.env',                  'Credenciales de BD'),
]

print('Verificando archivos necesarios:')
todos_ok = True
for archivo, descripcion in archivos_necesarios:
    existe = os.path.exists(archivo)
    icono  = '✅' if existe else '❌'
    print(f'  {icono}  {archivo:<25} {descripcion}')
    if not existe:
        todos_ok = False

print()
if todos_ok:
    print('✅ Todo listo para construir la imagen')
else:
    print('❌ Faltan archivos — revisa la lista antes de continuar')

---
## 🏗️ CELDA 4 — Construir la imagen Docker

```bash
docker build -t api-bsg .
```

- `-t api-bsg` → nombre (tag) de la imagen
- `.` → busca el `Dockerfile` en el directorio actual

> ⏱️ La primera vez tarda ~2 minutos. Las siguientes son casi instantáneas.

In [ ]:
# Construir la imagen
# Si ya existe, Docker la reconstruye solo las capas que cambiaron
print('Construyendo imagen api-bsg...')
print('(Puede tardar ~2 minutos la primera vez)\n')

code = run('docker build -t api-bsg .')

print()
if code == 0:
    print('✅ Imagen construida exitosamente')
else:
    print('❌ Error al construir — revisa el output de arriba')

In [ ]:
# Verificar que la imagen existe
print('Imágenes disponibles en tu máquina:')
run('docker images')

---
## 🚀 CELDA 5 — Correr el contenedor

```bash
docker run -d --name mi-api -p 8001:8001 --env-file .env api-bsg
```

| Flag | Qué hace |
|---|---|
| `-d` | Corre en background (detached) — no bloquea la terminal |
| `--name mi-api` | Nombre del contenedor para referenciarlo fácil |
| `-p 8001:8001` | Puerto local 8001 → puerto del contenedor 8001 |
| `--env-file .env` | Pasa las variables de entorno sin copiarlas a la imagen |
| `api-bsg` | La imagen a usar |

> 💡 **¿Por qué puerto 8001 y no 8000?**
> El 8000 es donde corre la API local (sin Docker). El 8001 es el contenedor.
> Así puedes tener ambos corriendo y distinguir cuál es cuál.

In [ ]:
# Primero eliminamos el contenedor si ya existía de una corrida anterior
run('docker stop mi-api 2>/dev/null; docker rm mi-api 2>/dev/null')

# Corremos el contenedor
print('Iniciando contenedor mi-api...')
code = run('docker run -d --name mi-api -p 8001:8001 --env-file .env api-bsg')

import time
time.sleep(2)  # Esperamos que arranque

print()
if code == 0:
    print('✅ Contenedor iniciado')
else:
    print('❌ Error al iniciar el contenedor')

In [ ]:
# Ver contenedores corriendo
print('Contenedores activos:')
run('docker ps')

In [ ]:
# Ver los logs del contenedor — lo que imprimiría en la terminal
print('Logs del contenedor mi-api:')
run('docker logs mi-api')

---
## ✅ CELDA 6 — Verificar que la API del contenedor responde

In [ ]:
import httpx
import json
import time

# Puerto 8001 = contenedor Docker
# Puerto 8000 = proceso local (si lo tienes corriendo)
CONTENEDOR_URL = 'http://localhost:8001'

def pretty(r):
    print(f'Status: {r.status_code}')
    print(json.dumps(r.json(), indent=2, ensure_ascii=False))

# Esperamos un poco más si el contenedor está arrancando
for intento in range(5):
    try:
        r = httpx.get(f'{CONTENEDOR_URL}/health', timeout=3)
        print(f'✅ Contenedor respondiendo en intento {intento + 1}')
        pretty(r)
        break
    except Exception:
        print(f'  Intento {intento + 1}/5 — esperando...')
        time.sleep(2)
else:
    print('❌ El contenedor no respondió — revisa docker logs mi-api')

In [ ]:
# Probar un endpoint real con datos
r = httpx.get(f'{CONTENEDOR_URL}/resumen')
print('Resumen desde el contenedor Docker:')
pretty(r)

In [ ]:
# Comparar puerto local vs contenedor (si tienes ambos corriendo)
print('=== Puerto 8000 (proceso local) ===')
try:
    r_local = httpx.get('http://localhost:8000/health', timeout=2)
    print(f'alumno: {r_local.json().get("alumno", "(no tiene campo alumno)")}  ← proceso local')
except Exception:
    print('No hay proceso local corriendo en 8000')

print()
print('=== Puerto 8001 (contenedor Docker) ===')
r_docker = httpx.get('http://localhost:8001/health', timeout=2)
print(f'alumno: {r_docker.json().get("alumno", "(no tiene campo alumno)")}  ← contenedor Docker')

---
## 🗂️ CELDA 7 — Comandos Docker esenciales

Estos son los comandos que usarás el 90% del tiempo con Docker:

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║              COMANDOS DOCKER ESENCIALES                      ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  IMÁGENES                                                    ║
║  docker build -t nombre .    Construir imagen del Dockerfile ║
║  docker images               Ver imágenes locales            ║
║  docker rmi nombre           Eliminar una imagen             ║
║                                                              ║
║  CONTENEDORES                                                ║
║  docker run -d -p 8001:8001  Correr un contenedor            ║
║    --name nombre             con nombre y puerto mapeado     ║
║    --env-file .env           pasando variables de entorno    ║
║    imagen                                                    ║
║  docker ps                   Ver contenedores corriendo      ║
║  docker ps -a                Ver todos (incluso detenidos)   ║
║  docker stop nombre          Detener un contenedor           ║
║  docker rm nombre            Eliminar un contenedor          ║
║  docker logs nombre          Ver los logs del contenedor     ║
║  docker logs -f nombre       Seguir los logs en vivo         ║
║                                                              ║
║  INSPECCIÓN                                                  ║
║  docker exec -it nombre sh   Entrar al contenedor            ║
║  docker inspect nombre       Ver configuración completa      ║
║                                                              ║
║  LIMPIEZA                                                    ║
║  docker system prune         Eliminar todo lo no usado       ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")

---
## 🎯 CELDA 8 — Proyecto Final: checklist

Estos son los pasos que debes completar para tu entregable.

In [ ]:
import httpx

def verificar_proyecto():
    """Checklist automático del proyecto final."""
    checks = []

    # Check 1: Docker está corriendo
    import subprocess
    r = subprocess.run('docker info', shell=True, capture_output=True)
    checks.append(('Docker Desktop activo', r.returncode == 0))

    # Check 2: Contenedor mi-api está corriendo
    r = subprocess.run('docker ps --filter name=mi-api --format "{{.Names}}"',
                       shell=True, capture_output=True, text=True)
    checks.append(('Contenedor mi-api corriendo', 'mi-api' in r.stdout))

    # Check 3: API responde en puerto 8001
    try:
        r = httpx.get('http://localhost:8001/health', timeout=3)
        checks.append(('API responde en puerto 8001', r.status_code == 200))

        # Check 4: Health check tiene campo 'alumno' con nombre
        alumno = r.json().get('alumno', '')
        tiene_nombre = bool(alumno) and alumno != 'TU NOMBRE AQUÍ'
        checks.append((f'Campo alumno configurado: "{alumno}"', tiene_nombre))
    except Exception:
        checks.append(('API responde en puerto 8001', False))
        checks.append(('Campo alumno configurado', False))

    # Check 5: Dashboard corre en puerto 8501
    try:
        r = httpx.get('http://localhost:8501', timeout=3)
        checks.append(('Dashboard Streamlit corriendo', r.status_code == 200))
    except Exception:
        checks.append(('Dashboard Streamlit corriendo', False))

    # Check 6: Dashboard apunta al puerto 8001 (contenedor)
    with open('dashboard.py', encoding='utf-8') as f:
        codigo = f.read()
    apunta_a_contenedor = 'localhost:8001' in codigo
    checks.append(('Dashboard apunta al puerto 8001', apunta_a_contenedor))

    # Check 7: Dashboard tiene nombre del alumno
    tiene_caption = 'Desarrollado por:' in codigo
    checks.append(('Dashboard tiene nombre del alumno', tiene_caption))

    # Reporte
    print('=' * 50)
    print('  CHECKLIST DEL PROYECTO FINAL')
    print('=' * 50)
    todos_ok = True
    for nombre, resultado in checks:
        icono = '✅' if resultado else '❌'
        print(f'  {icono}  {nombre}')
        if not resultado:
            todos_ok = False
    print('=' * 50)
    if todos_ok:
        print('  🎉 Listo para el screenshot del entregable')
    else:
        print('  ⚠️  Completa los checks marcados con ❌ antes del screenshot')

verificar_proyecto()

---
## 🛑 CELDA 9 — Detener el contenedor al terminar

In [ ]:
# Ejecuta esta celda cuando termines la sesión
print('Deteniendo y eliminando el contenedor...')
run('docker stop mi-api')
run('docker rm mi-api')
print('\n✅ Contenedor eliminado')
print('   La imagen api-bsg sigue en tu máquina para la próxima vez.')
print('   Para eliminarla también: docker rmi api-bsg')

---
## 🗺️ CELDA 10 — El pipeline completo: versión final

Lo que construiste a lo largo del curso:

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║       PIPELINE COMPLETO — Python para DE · BSG 2026            ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  CAP 1 — Fundamentos                                             ║
║  CSV sucio  →  Python puro  →  Pandas ETL                       ║
║                                                                  ║
║  CAP 2 — Almacenamiento                                          ║
║  extraer()  →  transformar()  →  cargar_a_mysql()               ║
║                              →  cargar_a_parquet()              ║
║                              →  calcular_metricas()             ║
║       ↓                                                          ║
║  MySQL en Aiven (nube)  +  Parquet (local)                       ║
║                                                                  ║
║  CAP 3 — Exposición                                              ║
║  FastAPI → /health /resumen /transacciones /metricas             ║
║       ↓                                                          ║
║  Streamlit → KPIs + Gráficas + Tabla + Buscador                  ║
║                                                                  ║
║  CAP 4 — Despliegue  ← ESTÁS AQUÍ                               ║
║  Dockerfile  →  docker build  →  docker run                      ║
║       ↓                                                          ║
║  Contenedor corriendo en cualquier máquina                       ║
║  Dashboard consumiendo la API del contenedor                     ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
""")

---
## ✅ Resumen de lo que construiste hoy

| Concepto | Comando / Archivo | Para qué |
|---|---|---|
| Imagen | `docker build -t api-bsg .` | Empacar la API |
| Contenedor | `docker run -d -p 8001:8001 ...` | Correr la imagen |
| Dockerfile | `FROM`, `COPY`, `RUN`, `CMD` | Instrucciones de la imagen |
| Cache de capas | `requirements_api.txt` antes que `api.py` | Build rápido |
| Variables de entorno | `--env-file .env` | Credenciales sin entrar a la imagen |
| Mapeo de puertos | `-p 8001:8001` | Exponer el contenedor al exterior |

---
## 📸 Entregable del Proyecto Final

Un **screenshot** que muestre simultáneamente:

1. ✅ `docker ps` con `mi-api` en status **Up**
2. ✅ `curl localhost:8001/health` con **tu nombre** en el campo `alumno`
3. ✅ Dashboard en `localhost:8501` con **tu nombre** en el caption
4. ✅ KPIs del dashboard con **datos reales** visibles

**Nombre del archivo:** `proyecto_final_TuNombre.png`

**Enviar a:** el canal de la clase

---
> 🎉 **Felicidades** — construiste un pipeline de datos completo:
> desde un CSV sucio hasta una API dockerizada con dashboard en producción.
> Eso es exactamente lo que hace un Data Engineer en una empresa real.